# AllSortsHub Cartoon Studio — Wan 2.2 Episode 1 Colab Generator

This notebook generates Episode 1 locally in Google Colab and keeps completed clips in Google Drive so the job can be resumed after a disconnect.

### Important GPU note
- **T4 (15 GB):** this notebook uses the smaller **Wan 2.2 TI2V-5B quantized path** when available instead of trying to load the 14B A14B model.
- **L4/A100 (24 GB+):** the official Wan 2.2 I2V-A14B path can be used.
- The official A14B examples require roughly 24 GB VRAM even with offloading, so the old notebook was a bad match for a free T4.

The notebook also avoids the old `requirements.txt` installation path that caused the Colab metadata/egg_info failure.

In [ ]:
# 1. GPU / Python diagnostic
!nvidia-smi
import torch, shutil
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. In Colab choose Runtime > Change runtime type > GPU, then rerun this cell.')
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU:', GPU_NAME)
print('VRAM GB:', round(VRAM_GB, 1))
print('FFmpeg:', shutil.which('ffmpeg'))
if VRAM_GB < 8:
    raise RuntimeError('This GPU has less than 8 GB VRAM; this notebook is not configured for it.')
USE_T4_PATH = VRAM_GB < 20
print('Selected path:', 'T4/low-VRAM Wan2.2 TI2V-5B' if USE_T4_PATH else 'official Wan2.2 I2V-A14B')

In [ ]:
# 2. Mount Google Drive for persistent model weights and generated clips
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AllSortsHub-Wan2.2'
!mkdir -p "$BASE/models" "$BASE/generated" "$BASE/output"
print('Persistent folder:', BASE)

In [ ]:
# 3. Get the cartoon project and install a Colab-safe video backend
%cd /content
!rm -rf cartoon-studio
!git clone -q https://github.com/parth01/AllSortsHub-Cartoon-Studio.git cartoon-studio
!python -m pip install -q --upgrade pip setuptools wheel
# Hardware-adaptive Wan 2.2 backend. It can use quantized TI2V-5B on lower-VRAM GPUs.
!python -m pip install -q 'git+https://github.com/mkamranr/videogen.git'
!apt-get update -qq && apt-get install -y -qq ffmpeg
!videogen doctor --mode i2v || true

## 4. Select the generation backend

The low-VRAM path is intentionally used on a T4. It uses Wan 2.2 TI2V-5B rather than the much larger I2V-A14B model. TI2V-5B supports image-to-video as well as text-to-video.

If `videogen doctor` refuses the T4 configuration, **do not download the 28 GB A14B model**. The notebook will stop with the reason instead of wasting the Colab session.

In [ ]:
# 4.1 Check the actual plan before downloading anything large
import subprocess, sys
if USE_T4_PATH:
    MODEL_ID = 'wan22'
    VARIANT = 'q4_k_m'
    PRESET = 'draft'
    print('T4 path: Wan 2.2 TI2V-5B / GGUF Q4_K_M')
    subprocess.run(['videogen','plan','AllSortsHub cartoon test','--mode','i2v','--model',MODEL_ID,'--variant',VARIANT,'--preset',PRESET,'--width','832','--height','480','--frames','49','--steps','4','--allow-slow'], check=False)
else:
    MODEL_ID = 'Wan-AI/Wan2.2-I2V-A14B'
    VARIANT = None
    PRESET = 'balanced'
    print('24GB+ path: official Wan 2.2 I2V-A14B')

In [ ]:
# 4.2 Download only the model appropriate for this GPU
import os, subprocess
MODEL_DIR = os.path.join(BASE, 'models')
if USE_T4_PATH:
    # videogen manages the quantized Wan 2.2 TI2V-5B assets.
    subprocess.run(['videogen','models','pull','wan22'], check=True)
else:
    from modelscope import snapshot_download
    A14B_DIR = os.path.join(MODEL_DIR, 'Wan2.2-I2V-A14B')
    os.makedirs(A14B_DIR, exist_ok=True)
    snapshot_download('Wan-AI/Wan2.2-I2V-A14B', local_dir=A14B_DIR)
print('Model preparation complete.')

In [ ]:
# 5. Prepare Episode 1 files
from pathlib import Path
ROOT = Path('/content/cartoon-studio/master-version/AllSortsHub-Billion 2')
GEN = Path(BASE) / 'generated'
LOCAL_GEN = ROOT / 'wan_i2v' / 'generated'
LOCAL_GEN.mkdir(parents=True, exist_ok=True)
GEN.mkdir(parents=True, exist_ok=True)
import json
with open(ROOT / 'wan_i2v' / 'manifest.json') as f: manifest = json.load(f)
# Restore prior completed clips from Drive into the project.
for p in GEN.glob('shot_*.mp4'):
    target = LOCAL_GEN / p.name
    if not target.exists() or target.stat().st_size < 10000:
        target.write_bytes(p.read_bytes())
print('Shots:', len(manifest['shots']))
print('Existing local clips:', len(list(LOCAL_GEN.glob('shot_*.mp4'))))

In [ ]:
# 6. Resumable Episode 1 generation
# Existing clips are skipped and copied to Drive after each successful shot.
import re, subprocess, shutil, time
prompts = (ROOT / 'wan_i2v' / 'prompts.txt').read_text()
STYLE = 'Modern 2D cel-shaded cartoon animation, bold clean black linework, semi-flat shading, vibrant colors, expressive anime-influenced facial acting, preserve the exact character designs and environment in the input image. Smooth readable hand-drawn motion. Keep faces, hair, clothing, proportions, props and background layout consistent.'
NEG = 'No photorealism, no 3D CGI, no live action, no extra fingers, no duplicate limbs, no warped faces, no character morphing, no costume changes, no hairstyle changes, no background replacement, no random objects, no random text, no logos, no watermark, no scene cuts, no sudden camera spins, no extreme deformation.'
def prompt_for(n):
    marker = f'SHOT {n:02d} —'
    start = prompts.find(marker)
    if start < 0: raise RuntimeError('Missing prompt for ' + marker)
    end = prompts.find('\n\nSHOT ', start + 2)
    if end < 0: end = prompts.find('\n\nNEGATIVE', start + 2)
    section = prompts[start:end if end >= 0 else None].split('\n', 1)[1].strip()
    return f'{STYLE} {section} {NEG}'
def save_to_drive(path):
    target = GEN / path.name
    shutil.copy2(path, target)
def run_low_vram(image, prompt, out, seed):
    if out.exists() and out.stat().st_size > 10000:
        print('SKIP', out.name); return
    cmd = ['videogen','i2v',str(image),prompt,'--model','wan22','--variant','q4_k_m','--preset','draft','--width','832','--height','480','--frames','49','--steps','4','--seed',str(seed),'--output',str(out),'--allow-slow','--no-plan']
    subprocess.run(cmd, check=True)
def run_a14b(image, prompt, out, seed):
    if out.exists() and out.stat().st_size > 10000:
        print('SKIP', out.name); return
    WAN = Path('/content/Wan2.2')
    if not WAN.exists():
        subprocess.run(['git','clone','-q','https://github.com/Wan-Video/Wan2.2.git',str(WAN)],check=True)
        subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(WAN/'requirements.txt')],check=True)
    cmd = [sys.executable,str(WAN/'generate.py'),'--task','i2v-A14B','--size','832*480','--ckpt_dir',str(A14B_DIR),'--offload_model','True','--convert_model_dtype','--t5_cpu','--frame_num','81','--image',str(image),'--prompt',prompt,'--base_seed',str(seed),'--save_file',str(out)]
    subprocess.run(cmd, cwd=WAN, check=True)
for shot in manifest['shots']:
    n = int(shot['id']); duration = float(shot['duration']); image = ROOT / shot['image']; out = LOCAL_GEN / f'shot_{n:02d}.mp4'
    p = prompt_for(n)
    # The low-VRAM backend intentionally renders short chunks. Longer shots are extended from the last frame.
    if USE_T4_PATH:
        if duration <= 3.0:
            run_low_vram(image, p, out, 910000+n)
        else:
            a = LOCAL_GEN/f'shot_{n:02d}a.mp4'; b = LOCAL_GEN/f'shot_{n:02d}b.mp4'
            run_low_vram(image, p, a, 910000+n)
            if not b.exists() or b.stat().st_size < 10000:
                frame = LOCAL_GEN/f'shot_{n:02d}_continuation.jpg'
                subprocess.run(['ffmpeg','-y','-sseof','-0.08','-i',str(a),'-frames:v','1','-q:v','2',str(frame)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
                run_low_vram(frame, f'{STYLE} Continue exactly from this final frame. {p} {NEG}', b, 1010000+n)
                frame.unlink(missing_ok=True)
            subprocess.run(['ffmpeg','-y','-i',str(a),'-i',str(b),'-filter_complex','[0:v][1:v]concat=n=2:v=1:a=0[v]','-map','[v]','-c:v','libx264','-pix_fmt','yuv420p',str(out)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    else:
        run_a14b(image, p, out, 910000+n)
    if out.exists() and out.stat().st_size > 10000:
        save_to_drive(out)
        print('SAVED', out.name, '-> Drive')
print('Generation pass complete.')

In [ ]:
# 7. Assemble Episode 1
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!python3 wan_i2v/assemble_episode.py
!cp -f output/AllSortsHub_Episode_01_WAN_MASTER.mp4 "$BASE/output/"
!cp -f output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4 "$BASE/output/"
!ls -lh output/AllSortsHub_Episode_01_WAN_MASTER.mp4 output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4

## 8. If Colab disconnects

1. Reconnect to a GPU runtime.
2. Remount Google Drive.
3. Rerun the setup/model cells as needed.
4. Rerun the generation cell.

Completed MP4 clips in `MyDrive/AllSortsHub-Wan2.2/generated/` are restored and skipped. Do not delete that folder.

### If the T4 path is refused or OOMs
Switch Colab to an **L4 or A100 with 24 GB+ VRAM** and rerun from the GPU diagnostic cell. The notebook will automatically select the official A14B path on a 24 GB+ GPU.